# PEFT Fine-Tuning with LoRA

This notebook demonstrates Parameter-Efficient Fine-Tuning (PEFT) using LoRA on a small Hugging Face model for text generation.

In [ ]:
%pip install peft transformers datasets accelerate

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset

# Load model and tokenizer
model_name = "microsoft/DialoGPT-small"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# PEFT config
peft_config = LoraConfig(r=8, lora_alpha=32, target_modules=["c_attn"], lora_dropout=0.1)
model = get_peft_model(model, peft_config)

# Load dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1%]")

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Training
training_args = TrainingArguments(output_dir="./results", num_train_epochs=1, per_device_train_batch_size=1)
trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_dataset)
trainer.train()